In [8]:
# Imports
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import polars as pl
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [9]:
# Data Cleaning
bikes = pl.read_csv('https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/bikes.csv')

bikes = bikes.with_columns((pl.col('casual') + pl.col('registered')).alias('total'))

bikes = bikes.drop(['dteday', 'casual', 'registered'])

bikes = bikes.with_columns([
    (2 * np.pi * pl.col('hr') / 24.0).sin().alias('hr_sin'),
    (2 * np.pi * pl.col('hr') / 24.0).cos().alias('hr_cos')
]).drop('hr')

bikes_pro = bikes.to_dummies(columns=['season', 'weathersit'])

In [10]:
# Setting up the Model

target_col = 'total'
feature_cols = [c for c in bikes_pro.columns if c != target_col]

X = bikes_pro.select(feature_cols).to_numpy()
y = bikes_pro.select(target_col).to_numpy().ravel()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [11]:
# Models
model = keras.Sequential([
    layers.Input(shape=(X_train_scaled.shape[1],)),
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.2),

    layers.Dense(64, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.2),

    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='linear')  # Single linear output for regression
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='mean_squared_error',
    metrics=['mean_absolute_error', keras.metrics.RootMeanSquaredError(name='rmse')]
)

early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True
)

history = model.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=150,
    batch_size=64,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/150
1125/1125 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - loss: 68855.5781 - mean_absolute_error: 168.4362 - rmse: 262.4035 - val_loss: 28922.5645 - val_mean_absolute_error: 114.1198 - val_rmse: 170.0664
Epoch 2/150
1125/1125 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - loss: 31902.1250 - mean_absolute_error: 118.4211 - rmse: 178.6117 - val_loss: 26568.1543 - val_mean_absolute_error: 104.8637 - val_rmse: 162.9974
Epoch 3/150
1125/1125 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - loss: 30711.3965 - mean_absolute_error: 115.4208 - rmse: 175.2467 - val_loss: 25940.9336 - val_mean_absolute_error: 103.3634 - val_rmse: 161.0619
Epoch 4/150
1125/1125 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - loss: 29832.0312 - mean_absolute_error: 113.3475 - rmse: 172.7195 - val_loss: 26052.8984 - val_mean_absolute_error: 103.0295 - val_rmse: 161.4091
Epoch 5/150
1125/1125 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - loss: 29536.9902 - mean_absolute_error: 112.3297 - rmse: 171.8633 - val_loss: 25728.6270 - val_mean_absolute_error: 103.0024 - val_

In [12]:
# Output
test_metrics = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"Model Output")
print(f"Test MAE:  {test_metrics[1]:.2f} bikes")
print(f"Test RMSE: {test_metrics[2]:.2f} bikes")

Model Output
Test MAE:  98.82 bikes
Test RMSE: 156.70 bikes
